# Chest X-ray Pneumonia — Baseline Training (Kaggle)

**Before running:** Settings → Accelerator → GPU (T4/P100). Add Input → search ‘Chest X-Ray Pneumonia’ (paultimothymooney) → Add.

This notebook is self-contained (mirrors `../src/*.py` in this repo) so it runs with only the dataset attached — no need to upload the repo itself as a Kaggle dataset. After a real run, copy `results/` back into the repo and delete the placeholder metrics.

In [ ]:
import torch
print('torch', torch.__version__, 'cuda available:', torch.cuda.is_available())
DATA_ROOT = '/kaggle/input/chest-xray-pneumonia/chest_xray'
OUT_DIR = '/kaggle/working'


## 1. Build train/val/test CSVs
Pools `train/`+`val/` (the shipped `val/` is only 16 images) into a fresh stratified split; `test/` is left untouched as the held-out set.

In [ ]:
import random
from pathlib import Path
import pandas as pd

CLASSES = {'NORMAL': 0, 'PNEUMONIA': 1}

def list_images(split_dir):
    rows = []
    for cls_name, label in CLASSES.items():
        cls_dir = split_dir / cls_name
        if not cls_dir.is_dir():
            continue
        for p in sorted(cls_dir.glob('*')):
            if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}:
                rows.append((str(p), label))
    return rows

def stratified_split(rows, val_frac=0.15, seed=42):
    by_label = {0: [], 1: []}
    for r in rows:
        by_label[r[1]].append(r)
    rng = random.Random(seed)
    train_rows, val_rows = [], []
    for label, items in by_label.items():
        rng.shuffle(items)
        n_val = max(1, int(len(items) * val_frac))
        val_rows.extend(items[:n_val])
        train_rows.extend(items[n_val:])
    rng.shuffle(train_rows); rng.shuffle(val_rows)
    return train_rows, val_rows

data_root = Path(DATA_ROOT)
pooled = list_images(data_root / 'train') + list_images(data_root / 'val')
test_rows = list_images(data_root / 'test')
train_rows, val_rows = stratified_split(pooled)

for name, rows in [('train.csv', train_rows), ('val.csv', val_rows), ('test.csv', test_rows)]:
    df = pd.DataFrame(rows, columns=['path', 'Pneumonia'])
    df.to_csv(f'{OUT_DIR}/{name}', index=False)
    print(name, len(df), 'images —', int(df.Pneumonia.sum()), 'pneumonia /', int((df.Pneumonia==0).sum()), 'normal')


## 2. Dataset, transforms, model

In [ ]:
import numpy as np
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms, models
import torch.nn as nn

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

def build_transforms(img_size, train):
    ops = [transforms.Resize((img_size, img_size))]
    if train:
        ops += [transforms.RandomHorizontalFlip(), transforms.RandomRotation(5)]
    ops += [transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)]
    return transforms.Compose(ops)

class CXRDataset(Dataset):
    def __init__(self, csv_path, img_size, train):
        df = pd.read_csv(csv_path)
        self.label_columns = [c for c in df.columns if c != 'path']
        self.paths = df['path'].tolist()
        self.labels = df[self.label_columns].values.astype(np.float32)
        self.tf = build_transforms(img_size, train)
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert('RGB')
        return self.tf(img), torch.from_numpy(self.labels[i])

def build_model(name, n_labels):
    if name == 'densenet121':
        m = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
        m.classifier = nn.Linear(m.classifier.in_features, n_labels)
    else:
        m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        m.fc = nn.Linear(m.fc.in_features, n_labels)
    return m


## 3. Train baseline
DenseNet-121 at 224px — fine on a Kaggle T4/P100, not feasible on the 2GB laptop GPU this repo was developed on.

In [ ]:
import json
from torch.utils.data import DataLoader
import torch.optim as optim
from sklearn.metrics import roc_auc_score, average_precision_score

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10
MODEL_NAME = 'densenet121'

train_ds = CXRDataset(f'{OUT_DIR}/train.csv', IMG_SIZE, train=True)
val_ds = CXRDataset(f'{OUT_DIR}/val.csv', IMG_SIZE, train=False)
n_labels = len(train_ds.label_columns)

dl_train = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
dl_val = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_model(MODEL_NAME, n_labels).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

history, best_auroc = [], -1.0
for epoch in range(1, EPOCHS + 1):
    model.train(); train_loss = 0.0
    for xb, yb in dl_train:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward(); optimizer.step()
        train_loss += loss.item() * xb.size(0)
    train_loss /= len(train_ds)

    model.eval(); ys, ps = [], []
    with torch.no_grad():
        for xb, yb in dl_val:
            probs = torch.sigmoid(model(xb.to(device))).cpu().numpy()
            ys.append(yb.numpy()); ps.append(probs)
    ys, ps = np.vstack(ys), np.vstack(ps)
    auroc = roc_auc_score(ys, ps, average='macro')
    auprc = average_precision_score(ys, ps, average='macro')
    print(f'Epoch {epoch}/{EPOCHS}  train_loss={train_loss:.4f}  val_auroc={auroc:.4f}  val_auprc={auprc:.4f}')
    history.append({'epoch': epoch, 'train_loss': train_loss, 'auroc_macro': auroc, 'auprc_macro': auprc})
    if auroc > best_auroc:
        best_auroc = auroc
        torch.save({'model_state': model.state_dict(), 'model_name': MODEL_NAME,
                     'label_columns': train_ds.label_columns, 'img_size': IMG_SIZE},
                    f'{OUT_DIR}/best.pt')

with open(f'{OUT_DIR}/history.json', 'w') as f:
    json.dump(history, f, indent=2)
print('Best val AUROC:', best_auroc)


## 4. Evaluate: AUROC/AUPRC, confusion matrix, ROC, calibration (ECE)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, roc_curve

def collect_logits(model, dl):
    ys, logits = [], []
    model.eval()
    with torch.no_grad():
        for xb, yb in dl:
            logits.append(model(xb.to(device)).cpu().numpy())
            ys.append(yb.numpy())
    return np.vstack(ys), np.vstack(logits)

def fit_temperature(logits, targets):
    logits_t = torch.tensor(logits, dtype=torch.float32, device=device)
    targets_t = torch.tensor(targets, dtype=torch.float32, device=device)
    temperature = torch.nn.Parameter(torch.ones(1, device=device))
    opt = torch.optim.LBFGS([temperature], lr=0.05, max_iter=100)
    crit = nn.BCEWithLogitsLoss()
    def closure():
        opt.zero_grad(); loss = crit(logits_t / temperature, targets_t); loss.backward(); return loss
    opt.step(closure)
    return float(temperature.detach().cpu().item())

def ece(y_true, y_prob, n_bins=10):
    edges = np.linspace(0, 1, n_bins + 1); total = 0.0; n = len(y_true)
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (y_prob >= lo) & (y_prob < hi) if hi < 1 else (y_prob >= lo) & (y_prob <= hi)
        if mask.sum() == 0: continue
        total += (mask.sum()/n) * abs(y_true[mask].mean() - y_prob[mask].mean())
    return float(total)

test_ds = CXRDataset(f'{OUT_DIR}/test.csv', IMG_SIZE, train=False)
dl_test = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
y_val, logits_val = collect_logits(model, dl_val)
y_test, logits_test = collect_logits(model, dl_test)
probs_raw = 1 / (1 + np.exp(-logits_test))
temperature = fit_temperature(logits_val[:, :1], y_val[:, :1])
probs_cal = 1 / (1 + np.exp(-logits_test / temperature))

metrics = {
    'auroc_macro': roc_auc_score(y_test, probs_raw, average='macro'),
    'auprc_macro': average_precision_score(y_test, probs_raw, average='macro'),
    'temperature': temperature,
    'ece_before_calibration': ece(y_test[:, 0], probs_raw[:, 0]),
    'ece_after_calibration': ece(y_test[:, 0], probs_cal[:, 0]),
}
print(json.dumps(metrics, indent=2))
with open(f'{OUT_DIR}/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

y_pred = (probs_raw[:, 0] >= 0.5).astype(int)
cm = confusion_matrix(y_test[:, 0], y_pred)
fig, ax = plt.subplots(figsize=(4,4))
ax.imshow(cm, cmap='Blues')
for (i, j), v in np.ndenumerate(cm): ax.text(j, i, str(v), ha='center', va='center')
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(['Normal','Pneumonia']); ax.set_yticklabels(['Normal','Pneumonia'])
ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title('Confusion matrix')
fig.tight_layout(); fig.savefig(f'{OUT_DIR}/confusion_matrix.png', dpi=150)

fpr, tpr, _ = roc_curve(y_test[:, 0], probs_raw[:, 0])
fig, ax = plt.subplots(figsize=(4,4))
ax.plot(fpr, tpr, label=f"AUROC = {metrics['auroc_macro']:.3f}")
ax.plot([0,1],[0,1],'--',color='gray')
ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.set_title('ROC curve'); ax.legend()
fig.tight_layout(); fig.savefig(f'{OUT_DIR}/roc_curve.png', dpi=150)
plt.show()


## 5. Grad-CAM on a handful of test images

In [ ]:
import torch.nn.functional as F

def get_last_conv_layer(model, name):
    return model.features.norm5 if name == 'densenet121' else model.layer4[-1]

class GradCAM:
    def __init__(self, model, layer):
        self.activations = self.gradients = None
        layer.register_forward_hook(lambda m, i, o: setattr(self, 'activations', o.detach()))
        layer.register_full_backward_hook(lambda m, gi, go: setattr(self, 'gradients', go[0].detach()))
        self.model = model
    def __call__(self, x, class_idx=0):
        self.model.zero_grad()
        logits = self.model(x)
        logits[:, class_idx].sum().backward()
        weights = self.gradients.mean(dim=(2,3), keepdim=True)
        cam = F.relu((weights * self.activations).sum(dim=1))
        cam = cam - cam.amin(dim=(1,2), keepdim=True)
        cam = cam / (cam.amax(dim=(1,2), keepdim=True) + 1e-8)
        return cam

def overlay(pil_img, cam, alpha=0.4):
    cam_resized = np.array(Image.fromarray((cam*255).astype('uint8')).resize(pil_img.size, Image.BILINEAR)) / 255.0
    heat = plt.get_cmap('jet')(cam_resized)[:, :, :3]
    base = np.array(pil_img.convert('RGB')) / 255.0
    return np.clip((1-alpha)*base + alpha*heat, 0, 1)

cam_engine = GradCAM(model, get_last_conv_layer(model, MODEL_NAME))
tf = build_transforms(IMG_SIZE, train=False)
test_df = pd.read_csv(f'{OUT_DIR}/test.csv').sample(n=6, random_state=42)

import os
os.makedirs(f'{OUT_DIR}/gradcam_examples', exist_ok=True)
for _, row in test_df.iterrows():
    pil_img = Image.open(row['path']).convert('RGB')
    x = tf(pil_img).unsqueeze(0).to(device)
    with torch.enable_grad():
        prob = torch.sigmoid(model(x)).detach().cpu().numpy()[0][0]
        cam = cam_engine(x)[0].cpu().numpy()
    ov = overlay(pil_img, cam)
    fig, ax = plt.subplots(figsize=(4,4))
    ax.imshow(ov); ax.axis('off')
    ax.set_title(f"true={int(row['Pneumonia'])}  p(pneumonia)={prob:.2f}", fontsize=9)
    fig.tight_layout()
    fig.savefig(f"{OUT_DIR}/gradcam_examples/{Path(row['path']).stem}_cam.png", dpi=150)
    plt.close(fig)
print('Grad-CAM overlays written to', f'{OUT_DIR}/gradcam_examples')


## 6. Next steps

- Download `/kaggle/working/{metrics.json,confusion_matrix.png,roc_curve.png,gradcam_examples/,history.json}` and commit them into `results/` in the repo.
- Re-run with `MODEL_NAME = 'resnet18'` for the model-comparison table (report Phase/Month 2).
- Add class-imbalance handling, MC-dropout uncertainty, and an ablation over augmentation/resolution (report Phase 2 / Month 3).